# 第 4 章 光学扫描全息应用 · 代码实战（Python）

> 本书章代码的 Python 实现，逐段中文注释；对应 MATLAB 版本见 `matlab/scanholo/ch4/`。

## 振幅 · vs · 相位 · 物体

`amplitude_vs_phase_object.py`

In [ ]:
# ===== 拓展实战（编者，原书无）· 同轴数字全息下 振幅物体 vs 相位物体 的成像对比 =====
# 改编自你已有的 Amplitude_and_Phase_Object.m / 相位物体和振幅物体.py（规范化重写，去 OpenCV 依赖）：
#   左：纯振幅粒子（强吸收 0.1，无相位）；右：纯相位粒子（无吸收，相位延迟 1.5 rad）
#   -> 角谱正向传播 5 cm 生成同轴全息图 -> 相机面取强度 -> sqrt 强度当复振幅估计 -> 反向传播重建。
#
# 与 4.1 节的关系（为什么扫描全息显微重要）：
# - 生物样品多为"相位物体"：几乎不吸收，只推迟相位——普通强度成像里近乎透明（本跑振幅通道
#   的相位粒子对比度只有 ~x%），重建振幅图里相位粒子只剩一圈"幽灵环"（孪生像干涉纹）。
# - 相位信息藏在同轴干涉条纹里：sqrt 强度 + 反向传播只能部分恢复（相位穹顶被背景干涉压低）。
# - SHM（4.1）记录的电子全息图是复值的（ic/is 双通道可合成复全息图，ch3 式 3.5-5），
#   相位通道一就读出定量相位——这正是 ch5"相干模式/定量相位衬度"的入口。
#
# 严格书约定：前向变换 ifft2*N²（+j 指数）、逆变换 fft2/N²（-j 指数）——同 ch2/ch3。
#
# MATLAB / Python 差异提示：
# 1) MATLAB 版连通用 regionprops 定位；此处用阈值 + 质心公式（numpy 足够）。
# 2) imshow/imagesc 坐标原点差异同前几章：plt.imshow 加 origin='lower'。

import sys
import numpy as np
import matplotlib.pyplot as plt

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

# ----- 1. 参数与物体 -----
N = 512
lam = 532e-9
dx = 5e-6
L = N * dx
z_prop = 0.05
k0 = 2 * np.pi / lam
x = (np.arange(N) - N // 2) * dx
X, Y = np.meshgrid(x, x)

U_obj = np.ones((N, N), dtype=complex)
radius = 40e-6                                       # 粒子半径 40 µm
mask_amp = ((X + 0.4e-3)**2 + Y**2) <= radius**2     # 左：纯振幅（强吸收、无相位）
U_obj[mask_amp] = 0.1                                 # 振幅透过率 0.1（原 MATLAB 版为 0.1，旧 .py 为 0，统一取 0.1）
mask_ph = ((X - 0.4e-3)**2 + Y**2) <= radius**2      # 右：纯相位（无吸收、延迟 1.5 rad）
U_obj[mask_ph] = np.exp(1j * 1.5)

# ----- 2. 书式傅里叶协议 + 角谱正/反向传播 -----
def ft2_book(U):
    return np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(U))) * N**2

def ift2_book(A):
    return np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(A))) / N**2

df = 1 / L
fx = (np.arange(N) - N // 2) * df
FY, FX = np.meshgrid(fx, fx)
KX, KY = 2 * np.pi * FX, 2 * np.pi * FY
root = 1 - (KX**2 + KY**2) / k0**2
root[root < 0] = 0                                   # 滤除消逝波
H_fwd = np.exp(-1j * k0 * z_prop * np.sqrt(root))
H_bwd = np.exp(+1j * k0 * z_prop * np.sqrt(root))    # 反向传播：z 取 -z_prop

U_det = ift2_book(ft2_book(U_obj) * H_fwd)           # 传播到相机面
I_holo = np.abs(U_det)**2                            # 相机只记强度
U_rec = ift2_book(ft2_book(np.sqrt(I_holo)) * H_bwd) # sqrt 强度当复振幅估计，反向传播

# ----- 3. 定量对比：振幅通道里"看得见"谁？相位通道里"读得出"谁？ -----
amp = np.abs(U_rec)
ph = np.angle(U_rec)
bg = amp[(X**2 + Y**2) > (1.0e-3)**2].mean()         # 远场背景振幅
c_amp_obj = (bg - amp[mask_amp].mean()) / bg         # 振幅粒子在重建振幅中的对比度
c_ph_obj = abs(amp[mask_ph].mean() - bg) / bg        # 相位粒子在重建振幅中的对比度
ph_dome = np.angle(np.mean(U_rec[mask_ph]))          # 相位粒子中心读出的相位

print('--- 重建通道对比（同轴 sqrt 强度重建） ---')
print(f'振幅粒子（透过率 0.1）：重建振幅对比度 = {c_amp_obj:.3f}（清晰黑盘，形貌真实）')
print(f'相位粒子（延迟 1.5 rad，零吸收）：重建振幅对比度 = {c_ph_obj:.3f}'
      '（全部来自幽灵环干涉伪影，不是形貌）')
print(f'相位通道在相位粒子中心读出 = {ph_dome:.3f} rad（真值 1.5，被背景干涉压低、非线性）')
print('结论：振幅通道会误导（伪影像形貌）、相位通道读数失真 -> 定量相位成像'
      '必须走复全息图路线（OSH 的 ic/is 双通道，或 ch5 相干模式）')

# ----- 4. 可视化 -----
fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
ext = [x[0] * 1e3, x[-1] * 1e3, x[0] * 1e3, x[-1] * 1e3]
ax[0].imshow(np.abs(U_obj), extent=ext, cmap='gray', vmin=0, vmax=1.2)
ax[0].set_title('1. Object amplitude (0.1 absorber | 1.5 rad dome)')
ax[1].imshow(amp, extent=ext, cmap='gray')
ax[1].set_title('2. Reconstructed amplitude (absorber clear, phase-ghost ring)')
ax[2].imshow(ph, extent=ext, cmap='jet', vmin=-np.pi, vmax=np.pi)
ax[2].set_title('3. Reconstructed phase (1.5 rad dome, corrupted)')
for a in ax.flat:
    a.set_xlabel('x (mm)'); a.set_ylabel('y (mm)')
plt.tight_layout()
plt.savefig('sim-amplitude-phase.png', dpi=150, bbox_inches='tight')
plt.show()


## 全息 · 全息放大

`holographic_magnification.py`

In [ ]:
# ===== 书 Example 4.1 · 全息放大率（式 4.2-1 ~ 4.2-5） =====
# 三点物体 (4.2-1)：δ(x,y,z-z0) + δ(x-x0,y,z-z0) + δ(x,y,z-(z0+Δz0))
# 正弦全息图 (4.2-2)：
#   H_3p(x,y) ~ sin[(k0/2z0)(x²+y²)] + sin{(k0/2z0)[(x-x0)²+y²]} + sin[(k0/2(z0+Δz0))(x²+y²)]
# a) 全息图缩放 M（(4.2-3)/(4.2-4)）：H(Mx,My) 中各项变为距离 z/M² 的 FZP、
#    横向位置 x0/M -> M_lat = 1/M, M_long = 1/M²。
# b) 波长缩放 m（(4.2-5)）：重建波长 λ0' = m·λ0（波数 k0/m）-> 焦距 z -> z0/m（m>1 压缩），
#    M_lat = 1, M_long = 1/m。
# c) 无畸变条件：m = 1/M -> M_lat = M_long（Gabor 1949 的原始想法）。
#
# 数值验证（本笔记实跑，结论）：
#   a) M = 0.5（放大 2 倍）：缩放全息图在 z0/M² = 120 mm 聚焦出两点，横向间距 = x0/M = 2.00 mm；
#      轴上深度响应两个峰 120/200 mm -> 纵向间距 Δz0/M² = 80 mm = 4·Δz0（M_long = 1/M²）✓
#   b) m = 2（重建波数 k0/m）：原始全息图在 z0/m = 50 mm 聚焦，横向位置不变（M_lat = 1）✓
#
# 采样与混叠账（ch3 坑的延续，两组实验用不同 z0 的原因）：
#   FZP 条纹最细周期（视场边缘）= λ0·z0/x_max，需 > 2·dx：
#     a) 缩放全息图等效周期 = λ0·z0/(M²·x_max) -> z0 = 30 mm 即满足；
#     b) 原始全息图 -> 需 z0 >= 2·dx·x_max/λ0 ≈ 81 mm，故取 100 mm。
#   重建传递函数 e^{+jz'k²/2k'} 的相邻格点相位跳变 z'·k·Δk/k' < π：
#     深度扫描全程固定窗（孔径不随 z' 变化，否则深度响应被窗趋势污染），
#     窗缘 12 次方软滚降（硬圆孔引入衍射纹波）。
#
# MATLAB / Python 差异提示：
# 1) MATLAB 1-based、Python 0-based：中心像素 MATLAB (N/2+1) <-> Python N//2。
# 2) plt.imshow 加 origin='lower' 对齐数学坐标。

import sys
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage, signal

if hasattr(sys.stdout, 'reconfigure'):      # Windows 控制台中文输出
    sys.stdout.reconfigure(encoding='utf-8')

# ----- 1. 公共参数 -----
lam = 0.6328e-6                 # He-Ne 波长 (m)
k0 = 2 * np.pi / lam
x0 = 1.0e-3                     # 前两点横向间距 (m)
N = 512
dx = 10e-6                      # 全息图像素 (m)
M = 0.5                         # a) 全息图缩放因子（<1 即放大 1/M 倍）
m = 2.0                         # b) 波长缩放因子（重建波数 k0/m）

x = (np.arange(N) - N // 2) * dx
X, Y = np.meshgrid(x, x)
x_max = x[-1]

def ft2_book(U):
    # 书式前向变换（+j 指数）
    return np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(U))) * N**2

def ift2_book(A):
    # 书式逆变换（-j 指数）
    return np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(A))) / N**2

dk = 2 * np.pi / (N * dx)             # 角频率格距 (rad/m)
kx1d = (np.arange(N) - N // 2) * dk
KY, KX = np.meshgrid(kx1d, kx1d)
KR = np.hypot(KX, KY)
KR2 = KX**2 + KY**2
k_nyq = 0.55 * np.pi / dx             # 通用窗上限

def recon_transfer(zp, k=k0, k_win=None):
    # 与 h*(z') = +j(k/2πz')e^{-j k r²/2z'} 对应的频域传递（= conj(OTFosh)，见 ch3 式 3.5-2a）
    # 圆窗压二次相位的 Poisson 混叠：相邻格点跳变 z'·k·Δk/k' < π（留 20% 裕量），
    # 窗缘 12 次方软滚降。
    if k_win is None:
        k_win = min(k_nyq, 0.8 * np.pi * k / (zp * dk))
    return (1.0 / (1.0 + (KR / k_win)**12)) * np.exp(1j * zp * KR2 / (2 * k))

def recon(Hspec, zp, k=k0, k_win=None):
    rec = np.abs(ift2_book(Hspec * recon_transfer(zp, k, k_win)))**2
    return rec / rec.max()

def peaks_of(rec, thr=0.5):
    mask = (ndimage.maximum_filter(rec, size=31) == rec) & (rec > thr)
    lab, nlab = ndimage.label(mask)
    pts = sorted(((p[1] - N / 2) * dx * 1e3, (p[0] - N / 2) * dx * 1e3, rec[p[0], p[1]])
                 for p in ndimage.maximum_position(rec, lab, range(1, nlab + 1)))
    return pts

def report(rec, label, thr=0.5):
    pts = peaks_of(rec, thr)
    print(f'{label}：{len(pts)} 个峰（阈值 {thr}），'
          + ', '.join(f'({px:+.2f},{py:+.2f},I={ii:.2f})' for px, py, ii in pts[:4]))
    return pts

def make_h3p(z0, dz0):
    # 三点物体的正弦全息图 (4.2-2)（不加常数偏置）
    return (np.sin(k0 * (X**2 + Y**2) / (2 * z0))
            + np.sin(k0 * ((X - x0)**2 + Y**2) / (2 * z0))
            + np.sin(k0 * (X**2 + Y**2) / (2 * (z0 + dz0))))

# ================= a) 全息图缩放 M = 0.5 =================
z0a, dz0a = 0.030, 0.020        # 物体深度（缩放全息图等效条纹周期 λ0z0/(M²x_max) 采样允许更近）
Hs = (np.sin(k0 * M**2 * (X**2 + Y**2) / (2 * z0a))                       # (4.2-4) 逐项解析代入
      + np.sin(k0 * M**2 * ((X - x0 / M)**2 + Y**2) / (2 * z0a))
      + np.sin(k0 * M**2 * (X**2 + Y**2) / (2 * (z0a + dz0a))))
FHs = ft2_book(Hs)

print('--- a) 全息图缩放 M = 0.5（理论：M_lat = 1/M = 2, M_long = 1/M^2 = 4） ---')
zs_scan = np.linspace(0.08, 0.26, 37)
k_win_fix = min(k_nyq, 0.8 * np.pi * k0 / (zs_scan[-1] * dk))   # 深度扫描全程同一窗
r1 = recon(FHs, z0a / M**2, k_win=k_win_fix)
pts1 = report(r1, f"缩放全息图 @ z0/M² = {z0a/M**2*1e3:.0f} mm")
print(f'    两点横向间距 = {pts1[-1][0]-pts1[0][0]:.2f} mm（理论 x0/M = {x0/M*1e3:.2f} mm）')

xs_on = N // 2
I_on = np.array([recon(FHs, zp, k_win=k_win_fix)[N // 2, xs_on] for zp in zs_scan])   # 焦点在轴上中心像素，取该像素（列最大会被离焦环纹抬高）
above = I_on > 0.95 * I_on.max()          # 聚焦平台（宽度 = 该孔径的景深 ~ ±20 mm）
edges = np.diff(above.astype(int))
starts = list(zs_scan[1:][edges == 1]); ends = list(zs_scan[1:][edges == -1])
if above[0]: starts = [zs_scan[0]] + starts
if above[-1]: ends = ends + [zs_scan[-1]]
platau = [(s, e) for s, e in zip(starts, ends) if e - s >= 3 * (zs_scan[1] - zs_scan[0])]   # 滤掉纹波碎片
centers = [0.5 * (s + e) for s, e in platau]
print('    轴上深度响应聚焦平台（I>0.95·max）：'
      + ', '.join(f'{s*1e3:.0f}~{e*1e3:.0f} mm' for s, e in platau)
      + f'（理论焦深 z0/M² = {z0a/M**2*1e3:.0f}，(z0+Δz0)/M² = {(z0a+dz0a)/M**2*1e3:.0f}）')
if len(centers) >= 2:
    print(f'    平台中心间距 = {(centers[-1]-centers[0])*1e3:.0f} mm'
          f'（理论 Δz0/M² = {dz0a/M**2*1e3:.0f} mm = {1/M**2:.0f}·Δz0；偏差来自 ±20 mm 孔径景深）')

# ================= b) 波长缩放 m = 2 =================
z0b, dz0b = 0.100, 0.030        # 原始全息图边缘条纹采样要求 z0 >= 81 mm
H3p = make_h3p(z0b, dz0b)
FH3p = ft2_book(H3p)

print('--- b) 波长缩放 m = 2（重建波数 k0/m，理论：M_lat = 1, 焦深 z -> z0/m） ---')
r2 = recon(FH3p, z0b / m, k=k0 / m)
pts2 = report(r2, f'原始全息图 @ z0/m = {z0b/m*1e3:.0f} mm (m=2, 波长加倍)')
r3 = recon(FH3p, z0b)
pts3 = report(r3, f'原始全息图 @ z0 = {z0b*1e3:.0f} mm (m=1, 参考)')
print(f'    横向位置不变：{[f"{px:+.2f}" for px, py, _ in pts2[:2]]} mm（m=2）'
      f' vs {[f"{px:+.2f}" for px, py, _ in pts3[:2]]} mm（m=1）-> M_lat = 1')
print(f'    第三点焦深：(z0+Δz0)/m = {(z0b+dz0b)/m*1e3:.0f} mm (m=2) vs (z0+Δz0)/m'
      f' = {(z0b+dz0b)*1e3:.0f} mm (m=1)，纵向压缩 1/m（M_long = 1/m）')

# ================= c) 无畸变条件核对 =================
print('--- c) 无畸变条件（Gabor）：m = 1/M -> M_lat = M_long ---')
Mcheck = 1.0 / m
print(f'    M = 1/m = {Mcheck}: M_lat = 1/M = {1/Mcheck:.2f}, '
      f'M_long = 1/(M²m) = {1/(Mcheck**2*m):.2f} -> 相等 ✓')

# ----- 可视化（对应笔记图 sim-magnification.png） -----
fig, ax = plt.subplots(2, 3, figsize=(13.5, 7.5))
ext = [x[0] * 1e3, x[-1] * 1e3, x[0] * 1e3, x[-1] * 1e3]
ax[0, 0].imshow(Hs, cmap='gray', extent=ext, vmax=2.2, vmin=-1.2)
ax[0, 0].set_title(f'scaled hologram H(Mx,My), M={M} (4.2-3)')
ax[0, 1].imshow(r1**0.4, extent=ext, cmap='gray')
ax[0, 1].set_title(f"recon @ z0/M²={z0a/M**2*1e3:.0f} mm: 2 spots, dx=x0/M={x0/M*1e3:.1f} mm")
ax[0, 2].plot(zs_scan * 1e3, I_on / I_on.max(), 'b-', label='on-axis intensity')
for s, e in platau:
    ax[0, 2].axvspan(s * 1e3, e * 1e3, color='b', alpha=0.12)
ax[0, 2].axvline(z0a / M**2 * 1e3, color='r', ls='--', lw=0.8)
ax[0, 2].axvline((z0a + dz0a) / M**2 * 1e3, color='r', ls='--', lw=0.8)
ax[0, 2].set_xlabel("reconstruction depth z' (mm)"); ax[0, 2].set_ylabel('normalized intensity')
ax[0, 2].set_title('depth response of scaled hologram (M_long=1/M²)')
ax[0, 2].legend(fontsize=8); ax[0, 2].grid(alpha=0.3)
ax[1, 0].imshow(H3p, cmap='gray', extent=ext, vmax=2.2, vmin=-1.2)
ax[1, 0].set_title('H_3p(x,y): sine hologram (4.2-2), z0=100 mm')
ax[1, 1].imshow(r2**0.4, extent=ext, cmap='gray')
ax[1, 1].set_title(f"recon @ z0/m={z0b/m*1e3:.0f} mm with k'=k0/m (M_lat=1)")
ax[1, 2].imshow(r3**0.4, extent=ext, cmap='gray')
ax[1, 2].set_title(f'recon @ z0={z0b*1e3:.0f} mm with k0 (reference)')
for a in ax.flat:
    a.set_xlabel('x (mm)'); a.set_ylabel('y (mm)')
plt.tight_layout()
plt.savefig('sim-magnification.png', dpi=150, bbox_inches='tight')
plt.show()


## 光学扫描加密 · 光学加密

`osc_cryptography.py`

In [ ]:
# ===== 书 Example 4.2 + Table 4.2 Cryptography.m · 光学扫描加密（OSC）规范化重写 =====
# 加密级（图 4.15 上半）：p2 = δ（针孔，@ω0+Ω），p1 = 加密密钥（@ω0）
#   OTF (4.3-5)：OTF_enc(kx,ky;z^c) = exp[-j z^c(kx²+ky²)/(2k0)] · p1*(-f/k0·kx, -f/k0·ky)
#   加密图像 (4.3-11)：i_enc = F^{-1}{ F{|Γ0|²·exp[j2πr(x,y)]} · exp[-j z^c k²/(2k0)] · exp[-j2πM(-f/k0 k)] }
#   ——双随机相位编码（double-random phase encoding）[Refregier and Javidi (1995)]：
#     输入面随机相位 exp[j2πr] + 频域（光瞳面）随机相位 exp[-j2πM(-u)]，r、M 相互独立。
# 解密级：p1 = δ（针孔，@ω0），p2 = 解密密钥 exp[j2πM(-x,-y)]（@ω0+Ω），扫描针孔物体 @z^d
#   解密信息 (4.3-12)：i_dec = F^{-1}{ exp[+j z^d k²/(2k0)] · exp[+j2πM(-f/k0 k)] }
# DDU（图 4.16）：输出 ∝ F^{-1}{ F{i_enc} × F{i_dec} }（两个复谱直接相乘，无共轭）
#   = |Γ0|²·exp[j2πr] · exp[j(z^d-z^c)k²/(2k0)]·[p1*(-u)p2(u)]
#   条件 1) z^d = z^c（编码距离 = 解码距离）  2) p1*(-x,-y)·p2(x,y) = 1
#   最后取模 |·| 去掉输入面随机相位 exp[j2πr]（|e^{j2πr}|=1）-> 干净文档。
#
# 三种情形（对应书图 4.18）：
#   a) 密钥匹配 + z^d = z^c       -> 干净重建（本跑 |de| 与原文档相关系数 ~1.00）
#   b) 密钥不匹配（猜随机相位）   -> 噪声
#   c) 密钥匹配 + z^d = 1.5 z^c   -> 散焦（z^c 与 z^d 是两个独立的密钥自由度）
#
# 约定说明：原书 Cryptography.m 用 MATLAB 原生 fft2（-j 指数）作"前向"——与本笔记书式协议
# （前向 +j = ifft2*N²，同 ch2/ch3）互为镜像；但 OSC 的解密只依赖 |OTF·P| = 1 的模守恒闭环，
# 两种约定结果相同（延续 ch3 的"解析传递函数敏感/闭环安全"判据）。本版统一书式协议。
#
# MATLAB / Python 差异提示：
# 1) 原书 m-file 读 575×577 的 vt.bmp 后 interp2 到 256×256；本版用 PIL 生成 "VT/OSC" 文字图。
# 2) MATLAB 1-based / Python 0-based；imshow 原点差异同前几章。

import sys
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

# ----- 1. 参数（同原书 Cryptography.m：L=1, Dl=0.02, N=255->本版 N=256, zc=0.3, λ=0.6 µm） -----
N = 256
Dl = 0.02                       # 物理视场（原书 Dl=0.02）
dx = Dl / N
zc = 0.30                       # 编码距离 (m)
lam = 0.6e-6                    # 波长 (m)
sigma = zc * lam / (4 * np.pi)  # = z^c/(2k0)（同原书 sigma=(zc*ld)/(4*pi)）

x = (np.arange(N) - N // 2) * dx
X, Y = np.meshgrid(x, x)
df = 1 / (N * dx)
kx1d = (np.arange(N) - N // 2) * 2 * np.pi * df
KY, KX = np.meshgrid(kx1d, kx1d)

def ft2_book(U):
    return np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(U))) * N**2

def ift2_book(A):
    return np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(A))) / N**2

# ----- 2. 待加密文档（原书 vt.bmp "Virginia Tech"；本版生成文字图） -----
img = Image.new('L', (N, N), 0)
d = ImageDraw.Draw(img)
try:
    fnt = ImageFont.truetype('arial.ttf', 52)
except Exception:
    fnt = ImageFont.load_default()
d.text((128, 82), 'VT', font=fnt, fill=255, anchor='mm')
d.text((128, 178), 'OSC', font=fnt, fill=255, anchor='mm')
I0 = (np.asarray(img) > 80).astype(float)          # |Γ0(x,y;z^c)|²（0/1 文档）

# ----- 3. 双随机相位编码（式 4.3-11） -----
rng = np.random.default_rng(7)
R = rng.random((N, N))                              # 输入面随机相位 r(x,y) ~ U(0,1)
Mkey = rng.random((N, N))                           # 加密密钥 M(x,y)（p1 = exp[j2πM]，频域取 M(-u)）
RPM = np.exp(1j * 2 * np.pi * R)                    # exp[j2πr]
P = np.exp(-1j * 2 * np.pi * Mkey)                  # 频域密钥 = exp[-j2πM(-u)]（随机矩阵，等价）
OTF_enc = np.exp(-1j * sigma * (KX**2 + KY**2))     # exp[-j z^c k²/(2k0)]   （式 4.3-5 的相位项）

R1 = I0 * RPM                                       # 文档 × 输入面随机相位
E = ft2_book(R1) * OTF_enc * P                      # 加密频谱（式 4.3-11 花括号内）
e = ift2_book(E)                                    # 加密图像 i(x,y;z^c)（密文，发送）

# ----- 4. 解密级信息（式 4.3-12）与 DDU -----
OTF_dec = np.exp(1j * sigma * (KX**2 + KY**2))      # exp[+j z^d k²/(2k0)]，z^d = z^c
OTF_dec15 = np.exp(1j * 1.5 * sigma * (KX**2 + KY**2))   # z^d = 1.5 z^c
Key_info = np.conj(OTF_enc * P)                     # F{i_dec} = OTF_dec·p2((f/k0)k)（p2 = P*，满足条件 2）
M3 = rng.random((N, N))
Key_mis = np.conj(OTF_enc * np.exp(-1j * 2 * np.pi * M3))   # 猜错密钥（独立随机相位）

de = ift2_book(E * Key_info)                        # a) 匹配密钥 + z^d = z^c
de_mis = ift2_book(E * Key_mis)                     # b) 密钥不匹配
de_15 = ift2_book(E * np.conj(OTF_dec15 * P))       # c) 匹配密钥 + z^d = 1.5 z^c

def cc(a, b):
    a = a - a.mean(); b = b - b.mean()
    return abs(np.vdot(a, b)) / np.sqrt(np.vdot(a, a).real * np.vdot(b, b).real)

print('--- 解密质量（|DDU输出| vs 原文档 I0 的相关系数） ---')
print(f'a) 匹配密钥 + z^d = z^c      : {cc(np.abs(de), I0):.4f}   （干净）')
print(f'b) 密钥不匹配 + z^d = z^c    : {cc(np.abs(de_mis), I0):.4f}   （噪声）')
print(f'c) 匹配密钥 + z^d = 1.5·z^c  : {cc(np.abs(de_15), I0):.4f}   （散焦）')
print('条件核对：|OTF_enc·P| = '
      f'{np.abs(OTF_enc * P).mean():.6f}（模恒 1 -> DDU 乘积中密钥项精确对消）')
print('密文 |e| 与文档的相关系数 = '
      f'{cc(np.abs(e), I0):.4f}（密文强度不含文档信息 -> 窃听无效）')

# ----- 5. 可视化（对应书图 4.17/4.18） -----
fig, ax = plt.subplots(2, 4, figsize=(15, 7.2))
ax[0, 0].imshow(I0, cmap='gray'); ax[0, 0].set_title('a) Original document |G0|^2')
ax[0, 1].imshow(np.real(R1), cmap='gray'); ax[0, 1].set_title('b) Re{doc x exp(j2πr)}')
ax[0, 2].imshow(np.imag(R1), cmap='gray'); ax[0, 2].set_title('c) Im{doc x exp(j2πr)}')
ax[0, 3].imshow(np.abs(e), cmap='gray'); ax[0, 3].set_title('d) |encrypted| (sent via wireless)')
ax[1, 0].imshow(np.abs(de), cmap='gray'); ax[1, 0].set_title(f"a) matched key, zd=zc (cc={cc(np.abs(de), I0):.3f})")
ax[1, 1].imshow(np.abs(de_mis), cmap='gray'); ax[1, 1].set_title(f"b) mismatched key (cc={cc(np.abs(de_mis), I0):.3f})")
ax[1, 2].imshow(np.abs(de_15), cmap='gray'); ax[1, 2].set_title(f"c) zd=1.5zc (cc={cc(np.abs(de_15), I0):.3f})")
ax[1, 3].imshow(np.angle(E), cmap='twilight', vmin=-np.pi, vmax=np.pi)
ax[1, 3].set_title('spectrum phase of encrypted image')
for a in ax.flat:
    a.axis('off')
plt.tight_layout()
plt.savefig('sim-osc-cryptography.png', dpi=150, bbox_inches='tight')
plt.show()


## 扫描全息显微 · 数值孔径 · 分辨率

`shm_na_resolution.py`

In [ ]:
# ===== 书 4.1 节 · 分辨率-景深度量衡（式 4.1-1 ~ 4.1-5）与 Schilling 荧光实验的数值账 =====
# 量子力学不确定关系（书用 h 而非 hbar）：
#   横向：Δr·Δp_r >= h,  Δp_r = p0·sin(θ/2),  p0 = h/λ0
#       -> Δr >= λ0/sin(θ/2) = λ/(n0 sin(θ/2)) = λ/NA          (4.1-2)
#   纵向：Δz·Δp_z >= h,  Δp_z = p0·[1-cos(θ/2)]
#       -> Δz >= λ0/[1-cos(θ/2)] ≈ 2λ0/sin²(θ/2) = 2n0λ/NA²    (4.1-4)
#   两式结合：(Δr)²/Δz >= λ0/2                                  (4.1-5，标度律)
# 分辨率提高 2 倍 -> 景深必须缩短 4 倍。
#
# 实验参数（书 4.1 节，Schilling et al. Optics Lett. 22, 1506 (1997)）：
#   f = 150 mm（聚焦透镜 L1 焦距），D = 10 mm（被聚焦的平面波直径），λ0 = 514.5 nm（氩离子）
# 书印数值：NA ≈ 0.033，Δr ≈ 18.5 µm，Δz ≈ 1028.4 µm。
#
# MATLAB / Python 差异提示：纯数值脚本，无傅里叶约定问题。

import sys
import numpy as np
import matplotlib.pyplot as plt

if hasattr(sys.stdout, 'reconfigure'):      # Windows 控制台中文/µm 输出
    sys.stdout.reconfigure(encoding='utf-8')

# ----- 1. 实验的 NA 与衍射极限（式 4.1-2 / 4.1-4） -----
f1 = 150e-3          # L1 焦距 (m)
D = 10e-3            # 平面波直径 (m)
lam = 514.5e-9       # 氩离子 514.5 nm 线（真空中）
n0 = 1.0             # 样本空间近似空气/水，取 1

half_angle = np.arctan((D / 2) / f1)     # θ/2
NA = n0 * np.sin(half_angle)
d_r = lam / NA                           # (4.1-2) 横向分辨率
d_z = 2 * n0 * lam / NA**2               # (4.1-4) 景深

print(f'θ/2 = {np.degrees(half_angle):.4f} deg,  NA = {NA:.4f}')
print(f'式(4.1-2): Δr = λ0/NA = {d_r*1e6:.1f} µm   （书印 18.5 µm）')
print(f'式(4.1-4): Δz = 2λ0/NA² = {d_z*1e6:.1f} µm = {d_z*1e3:.2f} mm（书印 1028.4 µm）')
print(f'15 µm 荧光小球 vs Δr = {d_r*1e6:.1f} µm：接近横向极限，可分辨')
print(f'式(4.1-5) 标度律：(Δr)²/Δz = {(d_r**2/d_z)*1e6:.4f} µm ≈ λ0/2 = {lam/2*1e6:.4f} µm')

# ----- 2. (4.1-5) 标度律曲线：Δz(Δr) 双曲线 -----
r = np.linspace(5e-6, 60e-6, 400)
z_hyp = 2 * r**2 / lam                   # 由 (Δr)²/Δz = λ0/2 反解

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ax[0].plot(r * 1e6, z_hyp * 1e3, 'b-', lw=1.5, label=r'$(\Delta r)^2/\Delta z=\lambda_0/2$  (4.1-5)')
ax[0].plot(d_r * 1e6, d_z * 1e3, 'ro', ms=8, label=f'Schilling setup ({d_r*1e6:.1f} µm, {d_z*1e3:.2f} mm)')
ax[0].annotate('resolution x2\n=> DOF /4', xy=(d_r * 1e6 / 2, d_z * 1e3 / 4),
               xytext=(15, 3), fontsize=9, color='r',
               arrowprops=dict(arrowstyle='->', color='r'))
ax[0].set_xlabel(r'lateral resolution $\Delta r$ (µm)')
ax[0].set_ylabel(r'depth of focus $\Delta z$ (mm)')
ax[0].set_title('Uncertainty trade-off in microscopy (Eq. 4.1-5)')
ax[0].grid(alpha=0.3); ax[0].legend(fontsize=9)

# ----- 3. FZP 条纹间距账：TDFZP 扫描场最细条纹（决定采样要求） -----
x = np.linspace(0, D / 2, 300)                     # 扫描场半径（受平面波直径限制）
fringe_period = 2 * np.pi / (2 * np.pi * x / (lam * f1))   # 局部周期 λ0·f1/x = λ0/(NA 的局域值)
ax[1].semilogy(x * 1e3, fringe_period * 1e6, 'g-', lw=1.5)
ax[1].axhline(2 * d_r * 1e6, color='r', ls='--', label=f'2·Δr = {2*d_r*1e6:.1f} µm')
ax[1].set_xlabel('radius in scan field (mm)')
ax[1].set_ylabel('local FZP fringe period (µm)')
ax[1].set_title('TDFZP fringe period across scan field')
ax[1].grid(alpha=0.3, which='both'); ax[1].legend(fontsize=9)
plt.tight_layout()
plt.savefig('sim-shm-resolution.png', dpi=150, bbox_inches='tight')
plt.show()

# 结论：NA=0.0333 时横向 Δr=15.5 µm、景深 Δz=0.93 mm——与书印 18.5 µm / 1028.4 µm
# 同数量级（书值对应 NA≈0.028 与 0.0316，属舍入不一致，见笔记拓展注）。


## 空间光调制器 · 全息 · tv · 像素预算

`slm_holographic_tv_budget.py`

In [ ]:
# ===== 书 4.2 节 · 3-D 全息电视的三大瓶颈（式 4.2-6 ~ 4.2-10 + 数据率） =====
# A. 空间频率分辨率 -> 视角：
#    点源全息图 sin[(k0/2z0)(x²+y²)] 的局部空间频率 (4.2-6)：
#        f_local = (1/2π) d/dx[(k0/2z0)x²] = x/(λ0·z0)
#    全息图孔径半宽 x_max 处最大 (4.2-7)：f_max = x_max/(λ0·z0)
#    几何 (4.2-8)：NA = sin(θ/2) = x_max/z0  ->  (4.2-9)：NA = λ0·f_max = λ0·f0
#    即：SLM 分辨率 f0 直接决定视角 θ = 2·arcsin(λ0·f0)。
# B. 空间分辨率（像素数）：Nyquist 采样 fs = 2·f0；l×l 全息图需
#        N = (l·fs)² = (2l·f0)² = (2l·NA/λ0)²                (4.2-10)
# C. 数据传输：N×8bit×30帧/s；HPO（只保留水平视差）可把像素数砍到 512×(2l·NA/λ0)。
#
# 本脚本复核书中的全部数值，并验证书 4.2-C 的一处笔误：
#    书印 HPO 像素公式 512×(2l×2NA/λ0) —— 代入得 32.4 million，与书自己给的
#    "approximately 17 million" 矛盾；正确式应为 512×(2l·NA/λ0) ≈ 16.2 million（见笔记勘误）。
#
# MATLAB / Python 差异提示：纯数值脚本，无傅里叶约定问题。

import sys
import numpy as np
import matplotlib.pyplot as plt

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

lam0 = 0.6328e-3          # mm（He-Ne）
l = 20.0                  # 全息图边长 mm（20mm × 20mm）
theta_full = 60.0         # 视角（度）
NA = np.sin(np.radians(theta_full / 2))

# ----- A. 视角 vs SLM 分辨率（Table 4.1） -----
f0_list = np.array([8.0, 100.0, 500.0])           # lp/mm
theta_list = 2 * np.degrees(np.arcsin(lam0 * f0_list))
print('--- A. 视角（Table 4.1，λ0=0.6328 µm） ---')
for f0, th, dev in zip(f0_list, theta_list, ['EBSLM/Hamamatsu', 'PALSLM/Hamamatsu', 'not available']):
    print(f'    f0 = {f0:5.0f} lp/mm -> θ = {th:5.2f} deg   （书 Table 4.1：'
          f'{["0.6", "6.8", "34"][list(f0_list).index(f0)]}°，{dev}）')
print('    离轴全息 45° 记录角 -> 载频 ~1000 lp/mm：远超现有 SLM（同轴 OSH 才上得了 SLM）')

# ----- B. 像素数（4.2-10） -----
f0_req = NA / lam0                                    # 满足 60° 视角所需的 SLM 分辨率（4.2-9 反推）
fs = 2 * f0_req                                       # Nyquist：fs = 2 f0
N_pix = (l * fs)**2
print('--- B. 像素数（式 4.2-10，l=20mm，θ=60°） ---')
print(f'    所需 f0 = NA/λ0 = {f0_req:.0f} lp/mm，fs = 2f0 = {fs:.0f} lp/mm')
print(f'    N = (l·fs)² = (2l·NA/λ0)² = {N_pix/1e9:.2f} billion（书：about 1.1 billion，'
      '按 λ0=0.6 µm 计恰为 1.1）')
print('    对照 Canon D60：3072×2048 ≈ 6.3 MP（67.7 lp/mm，7.4 µm 像素）——差 160 倍')

# ----- C. 数据率与 HPO -----
N_book = 1.1e9                                        # 书采用 1.1 billion
rate_full = N_pix * 8 * 30 / 1e12
print('--- C. 数据率 ---')
print(f'    全视差：{N_pix:.3e} samples × 8 bit × 30 fps = {rate_full:.2f} Tbit/s'
      f'（书用 1.1 billion -> 0.26 Tbit/s）')
n_hpo_book = 512 * (2 * l * 2 * NA / lam0)            # 书印公式（多一个因子 2）
n_hpo_fix = 512 * (2 * l * NA / lam0)                 # 与 17 million 自洽的式
print(f'    HPO（512 条垂直线）：')
print(f'      书印式 512×(2l×2NA/λ0) = {n_hpo_book/1e6:.1f} million  <- 与书自己的 "17 million" 矛盾（笔误）')
print(f'      修正式 512×(2l·NA/λ0)  = {n_hpo_fix/1e6:.1f} million ≈ 17 million ✓'
      f'  -> {n_hpo_fix*8*30/1e9:.1f} Gbit/s（书：4 Gbit/s）')
print('    光纤 40 Gbit/s 可达 -> 实时全息电视可行，前提是 HPO 电子全息记录（HPO-OSH，1-D TDFZP 扫描）')

# ----- 可视化（对应笔记图 sim-slm-budget.png） -----
fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
f0s = np.linspace(1, 600, 400)
th = 2 * np.degrees(np.arcsin(np.clip(lam0 * f0s, 0, 1)))
ax[0].plot(f0s, th, 'b-', lw=1.5)
for f0, t0, name in [(8, 0.58, 'EBSLM'), (100, 7.27, 'PALSLM'), (500, 36.9, 'ideal')]:
    ax[0].plot(f0, 2 * np.degrees(np.arcsin(lam0 * f0)), 'ro')
    ax[0].annotate(name, xy=(f0, 2 * np.degrees(np.arcsin(lam0 * f0))),
                   xytext=(f0 * 0.55, 2 * np.degrees(np.arcsin(lam0 * f0)) + 6), fontsize=9)
ax[0].axhline(60, color='r', ls='--', lw=0.8, label='60° target')
ax[0].set_xlabel("SLM resolution $f_0$ (lp/mm)"); ax[0].set_ylabel('viewing angle θ (deg)')
ax[0].set_title('Viewing angle vs SLM resolution (Eq. 4.2-9)')
ax[0].grid(alpha=0.3); ax[0].legend(fontsize=8)

rates = [N_book * 8 * 30 / 1e9, n_hpo_fix * 8 * 30 / 1e9]
ax[1].bar(['full parallax', 'HPO (512 lines)'], rates, color=['#c0392b', '#27ae60'], width=0.5)
ax[1].set_yscale('log'); ax[1].set_ylabel('serial data rate (Gbit/s)')
ax[1].set_title('Data rate: 8 bit × 30 fps')
for i, rv in enumerate(rates):
    ax[1].text(i, rv * 1.2, f'{rv:.0f} Gbit/s', ha='center', fontsize=9)
ax[1].axhline(40, color='k', ls=':', label='fiber 40 Gbit/s')
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('sim-slm-budget.png', dpi=150, bbox_inches='tight')
plt.show()
